In [6]:
from dataclasses import dataclass
from enum import Enum

from dotenv import load_dotenv
from openai import AsyncOpenAI
from pydantic import BaseModel

In [7]:
class StepStatus(str, Enum):
    PENDING = "pending"
    IN_PROGRESS = "in_progress"
    DONE = "done"
    FAILED = "failed"


@dataclass
class PlanStep:
    id: int
    description: str
    status: StepStatus = StepStatus.PENDING
    note: str = ""


@dataclass
class Plan:
    goal: str
    steps: list[PlanStep]

    def __str__(self) -> str:
        marks = {StepStatus.PENDING: "[ ]", StepStatus.IN_PROGRESS: "[>]", StepStatus.DONE: "[x]", StepStatus.FAILED: "[!]"}
        lines = []
        for step in self.steps:
            note = f" — {step.note}" if step.note else ""
            lines.append(f"{marks[step.status]} {step.id}. {step.description}{note}")
        return "\n".join(lines)


class PlanSchema(BaseModel):
    steps: list[str]

In [ ]:
load_dotenv()
client = AsyncOpenAI()
MODEL = "gpt-4.1-mini"

PLANNER_PROMPT = """
You are the planner for a browser agent. You only write the plan; a separate agent with a real browser will execute it.
Return 3 to 7 ordered steps. Each step is a page-level goal (e.g. "Log in"), not a single click or keystroke.
Do not invent credentials or values. If they are needed, add a step to find them on the page.
The last step should report what the task asks for.
"""


async def make_plan(client: AsyncOpenAI, task: str) -> Plan:
    response = await client.beta.chat.completions.parse(
        model=MODEL,
        messages=[
            {"role": "system", "content": PLANNER_PROMPT},
            {"role": "user", "content": task},
        ],
        response_format=PlanSchema,
    )
    parsed = response.choices[0].message.parsed
    steps = []
    for index, step_description in enumerate(parsed.steps, start=1):
        steps.append(PlanStep(id=index, description=step_description))
    return Plan(goal=task, steps=steps)

In [9]:
task = "Go to https://www.saucedemo.com/ and login, add to the Cart an item of $15.99 and go to Checkout, enter random test credentials and complete the purchase. Tell me what you see at the end"

plan = await make_plan(client, task)
print(plan)

[ ] 1. Open the website https://www.saucedemo.com/
[ ] 2. Log in using valid credentials found on the login page or a hint if provided
[ ] 3. Locate and add to the cart an item priced at $15.99
[ ] 4. Proceed to the cart and click the Checkout button
[ ] 5. Enter random test credentials in the checkout form and continue
[ ] 6. Complete the purchase
[ ] 7. Report the final confirmation or summary page content after purchase completion
